# 2. Send different beams through HTU with ImpactX

Magnets are tuned for a particular beam. Here we send two **synthetic Gaussian
beams**, at 100 MeV and 20 MeV, through the same HTU magnet settings.
They have the same sampled positions, angles, bunch length, charge and relative
energy spread. Neither beam comes from the WarpX example, and neither is
claimed to be an experimentally matched HTU beam.

This small calculation runs on the CPU. Open the notebook from the `htu` folder.

On the provided AWS session, select the **WarpX CPU** kernel for this notebook.

In [ ]:
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import openpmd_api as io
from scipy.constants import e

HTU = Path.cwd()
assert (
    HTU / "beamline_impactx/input_impactx.py"
).is_file(), "Open from the htu folder."
(HTU / "runs").mkdir(exist_ok=True)
run = Path(tempfile.mkdtemp(prefix="transport-", dir=HTU / "runs"))
energies_MeV = [100, 20]

## Run both beams

Each case runs in its own process and output folder. Magnet fields stay fixed.
Known PMQ (6 mm) and undulator focusing-magnet (4 mm) bore radii remove
particles that hit those modeled apertures. Other pipe/aperture dimensions
are not specified, so this is not a complete loss model of HTU.

In [ ]:
for energy in energies_MeV:
    case = run / f"{energy:g}MeV"
    with (run / f"{energy:g}MeV.log").open("w") as log:
        subprocess.run(
            [
                sys.executable,
                str(HTU / "beamline_impactx/input_impactx.py"),
                "--energy-MeV",
                str(energy),
                "--output",
                str(case),
            ],
            env=dict(os.environ, OMP_NUM_THREADS="2"),
            stdout=log,
            stderr=subprocess.STDOUT,
            check=True,
        )
print(f"Results: {run}")

## Watch the beam change, then compare survival

The density maps show the source, a screen after the PMQ triplet and a screen
in the chicane. Read each panel's axis units and ranges: the low-energy beam
spreads much more. Color shows charge per bin; it is not a common physical
density scale because the panel ranges differ.

Solid and dashed envelope curves show horizontal and vertical rms size.
A decrease in charge represents modeled aperture loss. If coordinates become
nonfinite, the plot stops at a dotted line: **the tracking model has become
invalid**, not demonstrated that every particle was physically lost.
In that case the final transmission is reported as `null`.

### Beam density at three positions

The following cell reads the saved particles and draws cross-sections of the beam.


In [ ]:
def read_screen(case, name):
    series = io.Series(
        str(Path(case) / "diags/openPMD" / f"{name}.h5"), io.Access.read_only
    )
    try:
        beam = series.iterations[min(series.iterations)].particles["beam"]
        position = float(beam.get_attribute("s_ref"))
        frame = beam.to_df().copy()
    finally:
        series.close()
    columns = ["position_x", "position_y", "weighting"]
    if not np.all(np.isfinite(frame[columns].to_numpy())):
        raise ValueError(
            f"{name} contains nonfinite coordinates; do not plot invalid tracking."
        )
    return frame, position


def plot_beam_density(
    run, energies=(100, 20), screens=("monitor", "TCPhosphor", "ChicaneSlit")
):
    import matplotlib.pyplot as plt
    from matplotlib.cm import ScalarMappable
    from matplotlib.colors import LogNorm

    labels = {
        "monitor": "Source",
        "TCPhosphor": "After PMQ triplet",
        "ChicaneSlit": "Mid-chicane",
    }
    fig, axes = plt.subplots(
        len(energies),
        len(screens),
        figsize=(12, 7),
        squeeze=False,
        constrained_layout=True,
    )
    for row, energy in enumerate(energies):
        for col, name in enumerate(screens):
            ax = axes[row, col]
            try:
                frame, position = read_screen(Path(run) / f"{energy:g}MeV", name)
            except ValueError:
                ax.text(
                    0.5,
                    0.5,
                    "Tracking invalid\nNo density shown",
                    ha="center",
                    color="firebrick",
                    transform=ax.transAxes,
                )
                ax.set_title(f"{energy:g} MeV · {labels.get(name,name)}")
                ax.set_axis_off()
                continue
            # µm at the source, mm downstream. Each panel fits the full beam;
            # explicitly label different scales instead of hiding the broad halo.
            scale, unit = (1e6, "µm") if name == "monitor" else (1e3, "mm")
            x, y = (
                frame.position_x.to_numpy() * scale,
                frame.position_y.to_numpy() * scale,
            )
            w = frame.weighting.to_numpy() * e * 1e12
            if len(x) == 0:
                ax.text(
                    0.5,
                    0.5,
                    "No surviving particles",
                    ha="center",
                    transform=ax.transAxes,
                )
                continue
            limit = max(float(np.max(np.abs(x))), float(np.max(np.abs(y))), 1e-6) * 1.03
            hist, xe, ye = np.histogram2d(
                x, y, bins=48, range=[[-limit, limit]] * 2, weights=w
            )
            # Charge per bin is easier to read than arbitrary normalized colors;
            # use the same logarithmic color scale in all panels.
            im = ax.pcolormesh(
                xe,
                ye,
                np.ma.masked_less_equal(hist.T, 0),
                cmap="viridis",
                norm=LogNorm(vmin=0.016, vmax=1.6),
                rasterized=True,
            )
            ax.set_aspect("equal")
            ax.set(
                xlabel=f"x ({unit})",
                ylabel=f"y ({unit})",
                title=f"{energy:g} MeV · {labels.get(name,name)}\ns = {position:.2f} m",
            )
            ax.text(
                0.03,
                0.96,
                f"{w.sum():.1f} pC",
                va="top",
                transform=ax.transAxes,
                bbox=dict(facecolor="white", alpha=0.85, edgecolor="none"),
                fontsize=9,
            )
    fig.colorbar(
        ScalarMappable(norm=LogNorm(vmin=0.016, vmax=1.6), cmap="viridis"),
        ax=list(axes.flat),
        label="Charge per bin (pC)",
        shrink=0.7,
        extend="max",
    )
    fig.suptitle(
        "Beam snapshots along HTU — read the axis scales in each panel", fontsize=15
    )
    return fig


fig = plot_beam_density(run, energies_MeV)
fig.savefig(run / "beam_density.png", dpi=160)
plt.show()

### Beam size and remaining charge

Read the measurements along the beamline. Stop each curve if tracking becomes invalid, so numerical failure is not shown as physical beam loss.


In [ ]:
def analyze(case):
    files = sorted((case / "diags").glob("reduced_beam_characteristics.*"))
    if not files:
        raise RuntimeError(f"No beam diagnostics in {case}")
    data = np.genfromtxt(files[0], names=True)
    data = np.atleast_1d(data)
    # Beam moments can be undefined after all particles are lost. Never call
    # nonfinite moments a measurement of loss; use the charge diagnostic.
    charge = np.abs(data["charge_C"])
    if not np.all(np.isfinite(charge)):
        raise RuntimeError(f"Nonfinite charge diagnostic in {case}")
    fraction = charge / 80e-12
    active = fraction > 0
    finite = np.isfinite(data["sigma_x"]) & np.isfinite(data["sigma_y"])
    bad = np.flatnonzero(active & ~finite)
    stop = int(bad[0]) if len(bad) else len(data)
    # Once the tracking map is invalid, its remaining-particle count is not
    # a trustworthy transmission prediction. Plot only the valid prefix.
    summary = dict(
        final_transmission=(None if len(bad) else float(fraction[-1])),
        first_nonfinite_s_m=(float(data["s"][stop]) if len(bad) else None),
        last_valid_transmission=float(fraction[max(0, stop - 1)]),
        max_rms_size_m=float(
            max(
                np.max(data["sigma_x"][:stop][(active & finite)[:stop]], initial=0),
                np.max(data["sigma_y"][:stop][(active & finite)[:stop]], initial=0),
            )
        ),
    )
    data = data[:stop]
    fraction = fraction[:stop]
    return data, fraction, summary


def plot_comparison(run, energies):
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True, constrained_layout=True)
    summaries = {}
    for energy in energies:
        case = run / f"{energy:g}MeV"
        data, fraction, summary = analyze(case)
        label = f"{energy:g} MeV"
        summaries[label] = summary
        (line,) = axes[0].plot(data["s"], 100 * fraction, label=label)
        valid = (
            (fraction > 0) & np.isfinite(data["sigma_x"]) & np.isfinite(data["sigma_y"])
        )
        # A missing envelope after extinction is not a beam of zero size.
        sx = np.where(valid, data["sigma_x"] * 1e3, np.nan)
        sy = np.where(valid, data["sigma_y"] * 1e3, np.nan)
        axes[1].plot(data["s"], sx, color=line.get_color(), label=f"{label}, x")
        axes[1].plot(data["s"], sy, "--", color=line.get_color(), label=f"{label}, y")
        if summary["first_nonfinite_s_m"] is not None:
            for ax in axes:
                ax.axvline(
                    summary["first_nonfinite_s_m"], color=line.get_color(), ls=":"
                )
            axes[0].annotate(
                f"{label}: tracking invalid beyond here",
                (summary["first_nonfinite_s_m"], 50),
                xytext=(8, 0),
                textcoords="offset points",
                fontsize=9,
            )

    axes[0].set(
        ylabel="Surviving charge (%)",
        ylim=(-2, 102),
        title="Same synthetic source geometry, different energy; fixed HTU magnets",
    )
    axes[1].set(
        ylabel="Surviving beam rms size (mm)", xlabel="Distance along beamline (m)"
    )
    axes[1].set_yscale("log")
    for ax in axes:
        ax.grid(alpha=0.25)
        ax.legend()
    fig.savefig(run / "comparison.png", dpi=160)
    plt.show()
    (run / "summary.json").write_text(json.dumps(summaries, indent=2) + "\n")
    return summaries


summaries = plot_comparison(run, energies_MeV)
print(json.dumps(summaries, indent=2))

## Try another energy

Change `energies_MeV` to `[100, 50]` and rerun from the setup cell. How do beam
size and survival change? Lower momentum makes the same magnets bend/focus
the beam more strongly. Good energy alone does not ensure correct matching.

For an optional connection to a plasma-generated bunch, see
[coupling instructions](README.md#optional-coupling).